# Repository Classifier — Explore the API

This notebook walks you through the library step by step: from a single classification call to custom classifiers, LLM-backed classification, and evaluation. Each section uses real GitHub repos so you can run the cells and see concrete results.

**Pipeline (under the hood):** Ground Truth → File-type inference → Heuristic or LLM. You only call one function; the library chooses the path.

## Step 1: One call, one result

The simplest way to classify a repo: pass a URL and a built-in classifier name. No API key. The result is a dict of **project type → confidence (0.0–1.0)**.

In [1]:
import repo_classifier
from repo_classifier import classify_repository_heuristic

print("Version:", repo_classifier.__version__)

# Laravel is a well-known PHP framework
results = classify_repository_heuristic(
    "https://github.com/laravel/laravel",
    "php",
    top_n=3,
)
print("Laravel (php):", results)

Version: 0.1.0
Laravel (php): {'Framework': 1.0, 'Library': 0.3466666666666667, 'Web App': 0.26666666666666666}


Try another repo and another built-in classifier. Same API; the classifier defines which project types are possible (e.g. PHP ecosystem vs Python ecosystem).

In [ ]:
from repo_classifier import classify_repository_heuristic

# Django: Python web framework
results = classify_repository_heuristic(
    "https://github.com/django/django",
    "python",
    top_n=3,
)
print("Django (python):", results)

# Express: JavaScript backend
results = classify_repository_heuristic(
    "https://github.com/expressjs/express",
    "javascript",
    top_n=3,
)
print("Express (javascript):", results)

Django (python): {'Web Framework': 1.0, 'API/Backend': 0.26595744680851063, 'Testing Tool': 0.2553191489361702}


ValueError: README not found for repository: https://github.com/expressjs/express

## Step 2: What is a classifier?

Built-in classifiers are exposed as `CLASSIFIERS.php`, `CLASSIFIERS.python`, `CLASSIFIERS.javascript`. You can use the **instance** instead of the string name; the result is the same. Each classifier has a fixed set of project types (and keyword weights) for that ecosystem.

In [ ]:
from repo_classifier import CLASSIFIERS, classify_repository_heuristic

print("Available built-ins:", CLASSIFIERS.names())
print("PHP project types (sample):", CLASSIFIERS.php.type_names[:5])

# Using the instance is equivalent to using the name "php"
results = classify_repository_heuristic(
    "https://github.com/laravel/laravel",
    CLASSIFIERS.php,
    top_n=3,
)
print("Laravel with CLASSIFIERS.php:", results)

## Step 3: Same repo, different lens

If you classify the same repo with different classifiers, you get different type sets. Example: Laravel with the PHP classifier yields PHP-specific types (Framework, Web App, …); with a minimal custom config we can focus on just two types and see how scores change.

In [ ]:
from repo_classifier import classify_repository_heuristic

# Custom config: only two types, with keywords and weights
custom = {
    "Web App": {"laravel": 10, "php": 5, "app": 3},
    "Framework": {"framework": 10, "artisan": 8, "composer": 6},
}
results = classify_repository_heuristic(
    "https://github.com/laravel/laravel",
    custom,
    top_n=2,
)
print("Laravel with custom (Web App vs Framework):", results)

## Step 4: LLM-based classification

When you need semantics rather than keywords, use `classify_repository_aimodel`. Same pipeline (ground truth → file-type → then LLM instead of heuristic). You must provide `model_name` (e.g. `openai/gpt-4o`, `deepseek/deepseek-chat`) and `api_key`. Copy `docs/.env.example` to `docs/.env` and set `REPO_CLASSIFIER_API_KEY` (and optionally `REPO_CLASSIFIER_MODEL_NAME`).

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from repo_classifier import CLASSIFIERS, classify_repository_aimodel

if Path("docs/.env").exists():
    load_dotenv("docs/.env")
elif Path(".env").exists():
    load_dotenv(".env")

api_key = os.getenv("REPO_CLASSIFIER_API_KEY")
model_name = os.getenv("REPO_CLASSIFIER_MODEL_NAME", "openai/gpt-4o")

if api_key:
    results = classify_repository_aimodel(
        repo_url="https://github.com/django/django",
        classifier=CLASSIFIERS.python,
        model_name=model_name,
        api_key=api_key,
        top_n=3,
    )
    print("Django (LLM):", results)
else:
    print("Set REPO_CLASSIFIER_API_KEY in docs/.env to run this cell.")

You can also pass a **list of type names** instead of a built-in classifier. The LLM will choose among those types only (no registration needed).

In [ ]:
from repo_classifier import classify_repository_aimodel
import os

if os.getenv("REPO_CLASSIFIER_API_KEY"):
    results = classify_repository_aimodel(
        repo_url="https://github.com/pallets/flask",
        classifier=["Web Framework", "Library", "CLI Tool", "API"],
        model_name=os.getenv("REPO_CLASSIFIER_MODEL_NAME", "openai/gpt-4o"),
        api_key=os.getenv("REPO_CLASSIFIER_API_KEY"),
        top_n=2,
    )
    print("Flask with custom type list:", results)
else:
    print("Set REPO_CLASSIFIER_API_KEY to run.")

## Step 5: Registry — use custom configs by name

Register a config so you can pass a **name** to `classify_repository_heuristic` or `classify_repository_aimodel` instead of the dict. Useful when you have several domain-specific classifiers (e.g. "game_dev", "data_science").

In [ ]:
from repo_classifier import (
    get_available_classifiers,
    get_classifier,
    register_classifier,
    unregister_classifier,
    classify_repository_heuristic,
)

print("Before:", get_available_classifiers())

register_classifier("game_dev", {
    "Game Engine": {"engine": 10, "game": 8, "render": 5},
    "Game Asset": {"sprite": 10, "texture": 8, "asset": 5},
})
print("After register:", get_available_classifiers())
print("get_classifier('game_dev') keys:", list(get_classifier("game_dev").keys()))

# Now you can classify by name
results = classify_repository_heuristic(
    "https://github.com/GodotEngine/godot",
    "game_dev",
    top_n=2,
)
print("Godot with 'game_dev':", results)

unregister_classifier("game_dev")
print("After unregister:", get_available_classifiers())

## Step 6: Ground truth and evaluation

Ground truth is a mapping **repo URL → project type**. The pipeline checks it first: if the URL is in ground truth, that type is returned with confidence 1.0. You can add entries in memory, save to JSON, load later, and **evaluate** a classifier (e.g. PHP) against that set to get accuracy and F1.

In [ ]:
from repo_classifier import (
    add_ground_truth_entry,
    get_ground_truth_repos,
    save_ground_truth,
    load_ground_truth,
    evaluate_classifier,
)

add_ground_truth_entry("https://github.com/laravel/laravel", "Framework")
add_ground_truth_entry("https://github.com/django/django", "Web Framework")
add_ground_truth_entry("https://github.com/expressjs/express", "Framework")
truth = get_ground_truth_repos()
print("Ground truth:", truth)

save_ground_truth("/tmp/demo_ground_truth.json", truth)
loaded = load_ground_truth("/tmp/demo_ground_truth.json")
print("Loaded from file:", loaded)

metrics = evaluate_classifier("php", truth)
print("evaluate_classifier('php', truth):", metrics)

## Step 7: Custom classifier from file or module

**From a text file:** use the format `TYPE: TypeName` then `keyword: weight` per line. `create_classifier_from_file(path)` returns a config dict; pass it to `classify_repository_heuristic` or `register_classifier`.

**From a Python module:** `load_classifier_from_module(module_path, attribute_name=None)` loads one or more configs from a module and registers them; then use the returned name(s) in classification.

In [ ]:
from pathlib import Path

from repo_classifier import (
    create_classifier_from_file,
    classify_repository_heuristic,
    register_classifier,
)

# Create a small config file (format: TYPE: Name, then keyword: weight)
config_path = Path("/tmp/demo_classifier.txt")
config_path.write_text(
    "TYPE: Backend\napi: 10\nserver: 5\nTYPE: Frontend\nui: 10\nweb: 5",
    encoding="utf-8",
)
config = create_classifier_from_file(str(config_path))
print("Config from file:", config)

results = classify_repository_heuristic(
    "https://github.com/django/django",
    config,
    top_n=2,
)
print("Django with file-based config:", results)

# Optional: register and use by name
register_classifier("from_file", config)
results2 = classify_repository_heuristic(
    "https://github.com/facebook/react",
    "from_file",
    top_n=2,
)
print("React with 'from_file':", results2)

**From a module:** if you have a Python file that defines a dict (e.g. `MY_TYPES = {...}`), call `load_classifier_from_module("path/to/file.py", "MY_TYPES")` to register it; then classify with the returned name.

In [ ]:
# Example (uncomment and set path to your module):
# from repo_classifier import load_classifier_from_module, classify_repository_heuristic
# name = load_classifier_from_module("path/to/classifiers.py", "MY_TYPES")
# results = classify_repository_heuristic("https://github.com/...", name, top_n=3)